In [ ]:
import json
import requests
import time
import os
import csv

INPUT_JSON = 'steam-game-list.json'
OUTPUT_FILE = 'steam_data_final_master.csv'
SLEEP_TIME = 1.6  

# 1. Load JSON App IDs
try:
    with open(INPUT_JSON, 'r', encoding='utf-8') as f:
        json_data = json.load(f)
    
    all_app_ids = [game['appid'] for game in json_data['response']['apps']]
    print(f"Loaded {len(all_app_ids)} App IDs from {INPUT_JSON}.")
except Exception as e:
    print(f"Error loading JSON: {e}")
    exit()

# 2. Setup CSV and Resume State
columns = [
    'AppID', 'Name', 'Price_USD', 'Initial_Price_THB', 
    'Windows', 'Mac', 'Linux', 
    'Genres', 'Total_Reviews', 'Positive_Reviews', 'Rating_Percent'
]

already_scraped = set()

# Check progress to allow pausing and resuming
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, mode='r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        for row in reader:
            already_scraped.add(int(row['AppID']))
    print(f"Found {len(already_scraped)} games already saved. Resuming from here.")
else:
    with open(OUTPUT_FILE, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(columns)
    print("Created new master CSV file.")

games_collected = len(already_scraped)
print("-" * 50)

# 3. The Extraction Loop
with open(OUTPUT_FILE, mode='a', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    
    for app_id in all_app_ids:
        # Skip if already saved
        if app_id in already_scraped:
            continue
            
        try:
            # --- A. FETCH API DATA ---
            # Call 1: Store API (US Currency for USD Price & General Data)
            store_url_us = f"https://store.steampowered.com/api/appdetails?appids={app_id}&cc=us"
            res_us = requests.get(store_url_us, timeout=15)
            
            # Call 2: Store API (TH Currency for Initial THB Price)
            store_url_th = f"https://store.steampowered.com/api/appdetails?appids={app_id}&cc=th"
            res_th = requests.get(store_url_th, timeout=15)
            
            # Call 3: Review API
            review_url = f"https://store.steampowered.com/appreviews/{app_id}?json=1&language=all"
            res_rev = requests.get(review_url, timeout=15)
            
            # --- B. HANDLE RATE LIMITS ---
            if res_us.status_code == 429 or res_th.status_code == 429 or res_rev.status_code == 429:
                print(f"\n Rate limit hit at {games_collected} games! Steam is blocking us. Sleeping for 5 minutes...")
                time.sleep(300)
                continue
                
            store_data_us = res_us.json()
            store_data_th = res_th.json()
            review_data = res_rev.json()
            
            # --- C. PROCESS DATA ---
            if store_data_us and str(app_id) in store_data_us and store_data_us[str(app_id)].get('success'):
                game_data = store_data_us[str(app_id)]['data']
                
                # Filter out non-games (DLCs, hardware, movies)
                if game_data.get('type') != 'game':
                    time.sleep(SLEEP_TIME)
                    continue
                    
                name = game_data.get('name', 'Unknown')
                is_free = game_data.get('is_free', False)
                
                # Get USD Price
                price_usd = 0.0
                if not is_free and 'price_overview' in game_data:
                    price_usd = game_data['price_overview'].get('final', 0) / 100.0
                
                # Get THB Initial Price
                price_thb = 0.0
                if store_data_th and str(app_id) in store_data_th and store_data_th[str(app_id)].get('success'):
                    th_data = store_data_th[str(app_id)]['data']
                    if not is_free and 'price_overview' in th_data:
                        price_thb = th_data['price_overview'].get('initial', 0) / 100.0
                elif is_free:
                    price_thb = 0.0
                else:
                    price_thb = price_usd # Fallback if TH API fails but US succeeds
                
                # Get Platforms
                platforms = game_data.get('platforms', {})
                win_support = platforms.get('windows', False)
                mac_support = platforms.get('mac', False)
                lin_support = platforms.get('linux', False)
                
                # Get Genres
                genres = ", ".join([g['description'] for g in game_data.get('genres', [])])
                
                # Get Reviews
                total_reviews = review_data.get('query_summary', {}).get('total_reviews', 0)
                positive_reviews = review_data.get('query_summary', {}).get('total_positive', 0)
                rating = (positive_reviews / total_reviews * 100) if total_reviews > 0 else 0
                
                # --- D. SAVE DATA ---
                writer.writerow([
                    app_id, name, price_usd, price_thb, 
                    win_support, mac_support, lin_support, 
                    genres, total_reviews, positive_reviews, round(rating, 2)
                ])
                file.flush() # Forces immediate write to disk
                
                games_collected += 1
                already_scraped.add(app_id)
                
                print(f"[{games_collected}] Saved: {name}")
                
            else:
                # API returned success: false (game might be delisted)
                print(f"[{games_collected}] Skipped (No Data): App {app_id}")
                
        except Exception as e:
            print(f"Error on App {app_id}: {e}. Skipping for now...")
            pass 
            
        # Respect API limits
        time.sleep(SLEEP_TIME)

print("\n dataset is ready: 'steam_data.csv'")